In [11]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import matplotlib.pyplot as plt

from modules import ForecastingModel

# Contents
1. [Load Dataset](#1-data-loading)
2. [Create nn module]
3. [Loss function and Optimizer]

## 1. Data Loading

In [2]:
stock_price_data_loader = ForecastingModel(
    price_type="Close",
    time_period="5y",
    time_interval="1wk"
)

ts = stock_price_data_loader.get_stock_price("CRWD")
ts = ts.to_numpy()

train_size = int(len(ts) * 0.7)
test_size = len(ts) - train_size
train, test = ts[:train_size], ts[train_size:]
print(len(train), len(test))

183 79


In [7]:
def create_dataset(dataset, lookback):
    '''
    Create lookback lag periods as features
    '''
    X, y = [], []
    for i in range(len(dataset)-lookback):
        feature = dataset[i:i+lookback]
        target = dataset[i+1:i+lookback+1]
        X.append(feature)
        y.append(target)
    X, y = np.array(X, dtype='f'), np.array(y, dtype='f')
    return torch.tensor(X), torch.tensor(y)

lookback = 52
X_train, y_train = create_dataset(train, lookback=lookback)
X_test, y_test = create_dataset(test, lookback=lookback)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([131, 52, 1]) torch.Size([131, 52, 1])
torch.Size([27, 52, 1]) torch.Size([27, 52, 1])


In [8]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


## 2. Create nn module

In [9]:
class LSTM_ForecastingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=50, num_layers=1, batch_first=True)
        self.linear = nn.Linear(50, 1)
        
    def forward(self, x):  # Forward is ran whenever we feed data into our model instance
        x, _ = self.lstm(x)
        x = self.linear(x)
        return x

In [10]:
model = LSTM_ForecastingModel()
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()
loader = DataLoader(
    TensorDataset(X_train, y_train), 
    shuffle=True, 
    batch_size=8
)
 
n_epochs = 2000
for epoch in range(n_epochs):
    model.train()  # Set instance to training mode
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)  # Forward pass by calling the model with inputs
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()  # Zero the gradient before backpropagation
        loss.backward()
        optimizer.step()  # Update parameters

    # Validation
    if epoch % 100 != 0:
        continue
    model.eval()
    with torch.no_grad():
        y_pred = model(X_train)
        train_rmse = np.sqrt(loss_fn(y_pred, y_train))
        y_pred = model(X_test)
        test_rmse = np.sqrt(loss_fn(y_pred, y_test))
    print("Epoch %d: train RMSE %.4f, test RMSE %.4f" % (epoch, train_rmse, test_rmse))

/var/folders/bl/8wwy1zb15dl1s95q42c0jx_m0000gp/T/ipykernel_15383/2929607567.py:26: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  train_rmse = np.sqrt(loss_fn(y_pred, y_train))
/var/folders/bl/8wwy1zb15dl1s95q42c0jx_m0000gp/T/ipykernel_15383/2929607567.py:28: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  test_rmse = np.sqrt(loss_fn(y_pred, y_test))


Epoch 0: train RMSE 182.4407, test RMSE 210.5695
Epoch 100: train RMSE 110.0410, test RMSE 135.1191
Epoch 200: train RMSE 64.4944, test RMSE 88.7899
Epoch 300: train RMSE 35.3221, test RMSE 61.4307
Epoch 400: train RMSE 20.2109, test RMSE 44.3858
Epoch 500: train RMSE 12.8203, test RMSE 32.8462
Epoch 600: train RMSE 10.1087, test RMSE 27.7497
Epoch 700: train RMSE 9.0139, test RMSE 25.2982
Epoch 800: train RMSE 8.0868, test RMSE 24.3161
Epoch 900: train RMSE 7.4007, test RMSE 23.8707
Epoch 1000: train RMSE 6.9738, test RMSE 24.0771
Epoch 1100: train RMSE 6.7150, test RMSE 22.5409
Epoch 1200: train RMSE 6.7167, test RMSE 22.4870
Epoch 1300: train RMSE 6.0715, test RMSE 22.9165
Epoch 1400: train RMSE 6.2637, test RMSE 23.4500
Epoch 1500: train RMSE 5.8316, test RMSE 22.7059
Epoch 1600: train RMSE 5.7298, test RMSE 25.3672
Epoch 1700: train RMSE 5.2827, test RMSE 25.6471
Epoch 1800: train RMSE 5.3377, test RMSE 28.0958
Epoch 1900: train RMSE 5.0692, test RMSE 28.0682


In [15]:
with torch.no_grad():
    # shift train predictions for plotting
    train_plot = np.ones_like(ts) * np.nan
    y_pred = model(X_train)
    y_pred = y_pred[:, -1, :]
    train_plot[lookback:train_size] = model(X_train)[:, -1, :]
    # shift test predictions for plotting
    test_plot = np.ones_like(ts) * np.nan
    test_plot[train_size+lookback:len(ts)] = model(X_test)[:, -1, :]

plt.plot(ts, c='b')
plt.plot(train_plot, c='r')
plt.plot(test_plot, c='g')
plt.show()

AttributeError: 'Tensor' object has no attribute 'copy'